## Setup

In [ ]:
import sys
import os
import importlib

from tabulate import tabulate
import matplotlib.pyplot as plt

sys.path.append("../src")

import utils
import plot
import rstats

# Reload modules to apply any changes
importlib.reload(utils)
importlib.reload(plot)
importlib.reload(rstats)

In [ ]:
FILENAME = os.getenv("FILENAME", "UFABC_PLT_combined")
BACKEND = os.getenv("BACKEND", "gemini")
DST = f"../results/{BACKEND}/{FILENAME}/"

print(f"{FILENAME=}")
print(f"{BACKEND=}")

df = utils.load(f"../results/{BACKEND}/{FILENAME}/metrics.csv")
print(f"{df.shape=}")

In [ ]:
tmp = ["id", "category", "concept"] if "category" in df.columns else ["id", "concept"]
grouped = df.groupby(tmp, as_index=False)
dfx = grouped.mean(numeric_only=True)
print(tabulate(dfx.head(3), headers="keys", showindex=False))

## Analysis

In [ ]:
metrics = [
    "entropy",
    "distance_next",
    "distance_centroid_static",
    "vel_magnitude",
    "acc_magnitude",
]

if "category" not in dfx.columns:
    dfx = dfx.rename(columns={"concept": "category"})

for i, metric in enumerate(metrics):
    print(f"[{i + 1}/{len(metrics)}] Analyzing '{metric}'")

    # Use lognormal for all metrics, except for distance_centroid_static
    family = "lognormal" if metric != "distance_centroid_static" else "gaussian"
    res, pred, pairs, stdout = plot.boxplot(dfx, metric, family=family)

    plt.savefig(f"{DST}/boxplot-glmm-{metric}.png", bbox_inches="tight")
    plt.close()

    with open(f"{DST}/r-output-{metric}.txt", "w") as fp:
        fp.write(stdout)